## **Imports, Custom Layer Definition, and Setup**

In [ ]:
import os
import cv2
import glob
import math
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from mtcnn import MTCNN
from datetime import datetime, date

# ── 1. ARCFACE CUSTOM LAYER ───────────────────────────────────────────────────
# Must match training notebook exactly — training=None argument is required
class ArcFaceLayer(tf.keras.layers.Layer):
    def __init__(self, num_classes, s=64.0, m=0.55, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.s = s
        self.m = m
        self.cos_m = tf.cast(math.cos(m), tf.float32)
        self.sin_m = tf.cast(math.sin(m), tf.float32)
        self.th    = tf.cast(math.cos(math.pi - m), tf.float32)
        self.mm    = tf.cast(math.sin(math.pi - m) * m, tf.float32)

    def build(self, input_shape):
        emb_dim = input_shape[0][-1]
        self.W = self.add_weight(
            name='arcface_weights',
            shape=(emb_dim, self.num_classes),
            initializer='glorot_uniform',
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs, training=None):   # training=None required
        embeddings, labels = inputs
        emb_norm = tf.nn.l2_normalize(embeddings, axis=1)
        W_norm   = tf.nn.l2_normalize(self.W, axis=0)
        cosine   = tf.matmul(emb_norm, W_norm)
        cosine   = tf.clip_by_value(cosine, -1.0 + 1e-7, 1.0 - 1e-7)
        if not training:
            return cosine * self.s   # pure cosine logits, no margin
        sine        = tf.sqrt(1.0 - tf.square(cosine))
        cos_theta_m = cosine * self.cos_m - sine * self.sin_m
        cos_theta_m = tf.where(cosine > self.th, cos_theta_m, cosine - self.mm)
        output      = labels * cos_theta_m + (1.0 - labels) * cosine
        return output * self.s

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_classes': self.num_classes, 's': self.s, 'm': self.m})
        return cfg


# ── 2. PATH CONFIGURATION ─────────────────────────────────────────────────────
MODEL_PATH   = r'C:\Users\User\Downloads\face_demo\model\arcface_finetuned.keras'
CLASS_PATH   = r'C:\Users\User\Downloads\face_demo\model\class_names.json'
DATABASE_DIR = r'C:\Users\User\Downloads\face_demo\deployment\database'
ATTENDANCE_DIR = r'C:\Users\User\Downloads\face_demo\deployment\attendance'

os.makedirs(DATABASE_DIR, exist_ok=True)
os.makedirs(ATTENDANCE_DIR, exist_ok=True)
detector = MTCNN()


# ── 3. LOAD FINE-TUNED MODEL ──────────────────────────────────────────────────
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'ArcFaceLayer': ArcFaceLayer},
    compile=False
)
with open(CLASS_PATH, 'r') as f:
    CLASS_NAMES = json.load(f)
NUM_CLASSES = len(CLASS_NAMES)
print(f'Model loaded — {NUM_CLASSES} fine-tuned classes')


# ── 4. BUILD 512-DIM EMBEDDING EXTRACTOR ──────────────────────────────────────
# We stop BEFORE ArcFaceLayer and take the 512-dim backbone output.
# This is still your fine-tuned model — same weights, just reading earlier.
# Print all layers to confirm 'dropout' is the correct layer name.
print('\nLayer map (find the 512-dim layer just before arcface_margin):')
for l in model.layers:
    try:
        print(f'  {l.name:45s}  {l.output.shape}')
    except Exception:
        pass

input_image = None
for inp in model.inputs:
    if inp.name.split(':')[0].endswith('input_image'):
        input_image = inp
        break
if input_image is None:
    input_image = model.inputs[0]

embedding_model = tf.keras.Model(
    inputs=input_image,
    outputs=model.get_layer('dropout').output   # 512-dim
)
print('\n✅ System ready!')
print(f'   Embedding size : 512-dim (from fine-tuned backbone)')
print(f'   Classes trained: {NUM_CLASSES}')


Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\User\Downloads\face_demo\venv\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\User\Downloads\face_demo\venv\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\User\Downloads\face_demo\venv\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.



Model loaded — 32 fine-tuned classes

Layer map (find the 512-dim layer just before arcface_margin):
  input_image                                    (None, 112, 112, 3)
  random_rotation                                (None, 112, 112, 3)
  ResNet34                                       (None, 512)
  bn_head                                        (None, 512)
  dropout                                        (None, 512)
  input_labels                                   (None, 32)
  arcface_margin                                 (None, 32)

✅ System ready!
   Embedding size : 512-dim (from fine-tuned backbone)
   Classes trained: 32


## **Pipline to Preprocessing & Math Helpers for the input**

In [2]:
# ── FACE ALIGNMENT (same pipeline as training) ────────────────────────────────
def align_and_crop_face_from_frame(frame_bgr, target_size=(112, 112)):
    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(img_rgb)
    if not results:
        return None   # ← return None so callers can skip no-face frames
    r = max(results, key=lambda x: x['confidence'])
    kp = r['keypoints']
    le, re = np.array(kp['left_eye']), np.array(kp['right_eye'])
    angle  = np.degrees(np.arctan2(re[1] - le[1], re[0] - le[0]))
    center = ((le[0] + re[0]) / 2, (le[1] + re[1]) / 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    h, w = img_rgb.shape[:2]
    rotated = cv2.warpAffine(img_rgb, M, (w, h))
    x, y, bw, bh = r['box']
    margin = int(max(bw, bh) * 0.10)
    x1, y1 = max(0, x - margin),      max(0, y - margin)
    x2, y2 = min(w, x + bw + margin), min(h, y + bh + margin)
    face = rotated[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return cv2.resize(face, target_size)   # RGB uint8


def preprocess_face(aligned_rgb):
    """ArcFace normalisation: [0,255] → [-1, 1]"""
    return (aligned_rgb.astype(np.float32) - 127.5) / 128.0


def get_embedding(aligned_rgb):
    """Run aligned face through fine-tuned backbone → 512-dim L2-normalised vector."""
    img_norm  = preprocess_face(aligned_rgb)
    img_batch = np.expand_dims(img_norm, axis=0)                        # (1,112,112,3)
    emb = embedding_model(img_batch, training=False).numpy()[0]         # (512,)
    return emb / np.linalg.norm(emb)                                    # L2-normalise


def load_registered_database():
    """Load all .npy gallery files from DATABASE_DIR."""
    database = {}
    for path in glob.glob(os.path.join(DATABASE_DIR, '*.npy')):
        name = os.path.basename(path).replace('.npy', '')
        database[name] = np.load(path)   # 512-dim vector
    print(f'Gallery loaded: {len(database)} registered user(s)')
    return database


# ── DAILY ATTENDANCE CSV LOGIC ────────────────────────────────────────────────
def get_today_csv_path():
    """Return path for today's attendance file, e.g. attendance_2026-05-30.csv"""
    today_str = date.today().strftime('%Y-%m-%d')
    return os.path.join(ATTENDANCE_DIR, f'attendance_{today_str}.csv')


def get_or_create_today_attendance(registered_users: dict) -> pd.DataFrame:
    """
    Load today's attendance CSV.
    If it does not exist → create a fresh one with every registered user set to Absent.
    If it already exists → load and return as-is (session already started today).
    """
    csv_path = get_today_csv_path()

    if not os.path.exists(csv_path):
        # Brand-new day — build a fresh sheet from the current gallery
        rows = [
            {
                'Student_Name':        name,
                'Registration_Status': 'Active',
                'Last_Seen_Time':      'Never',
                'Attendance_Status':   'Absent'   # everyone starts Absent
            }
            for name in registered_users.keys()
        ]
        df = pd.DataFrame(rows, columns=[
            'Student_Name', 'Registration_Status',
            'Last_Seen_Time', 'Attendance_Status'
        ])
        df.to_csv(csv_path, index=False)
        print(f'📅 New attendance sheet created: {os.path.basename(csv_path)}')
        print(f'   {len(rows)} student(s) initialised → Absent')
    else:
        df = pd.read_csv(csv_path)
        print(f'📂 Loaded existing sheet: {os.path.basename(csv_path)}')
        print(f'   {len(df)} student(s) found')

    return df


print('✅ All helper functions ready')


✅ All helper functions ready


# **1. User Registration**  (Enrollment)

**Registrater to System by Input Files Manually**

In [3]:
def register_user_from_files(user_name, image_paths_list):
    """
    Register a new user from static image files.
    Extracts 512-dim embeddings from the fine-tuned backbone.
    Saves mean embedding as user_name.npy in DATABASE_DIR.
    Adds user to today's attendance CSV as Absent.
    """
    print(f'\n--- Registering: {user_name} ---')
    collected = []

    for img_path in image_paths_list:
        if not os.path.exists(img_path):
            print(f'  ⚠️  Not found: {img_path}')
            continue
        frame_bgr = cv2.imread(img_path)
        if frame_bgr is None:
            print(f'  ⚠️  Cannot decode: {img_path}')
            continue
        aligned = align_and_crop_face_from_frame(frame_bgr)
        if aligned is None:
            print(f'  ⚠️  No face detected: {os.path.basename(img_path)}')
            continue
        emb = get_embedding(aligned)   # 512-dim L2-normalised
        collected.append(emb)
        print(f'  ✅  {os.path.basename(img_path)}  →  emb shape {emb.shape}')

    if not collected:
        print('❌ Registration failed — no valid faces found.')
        return

    # Mean embedding, re-normalise
    master = np.mean(collected, axis=0)
    master = master / np.linalg.norm(master)

    # Save 512-dim vector
    save_path = os.path.join(DATABASE_DIR, f'{user_name}.npy')
    np.save(save_path, master)
    print(f'\n✨ Saved 512-dim template for "{user_name}" → {save_path}')

    # Add to today's attendance sheet if not already there
    csv_path = get_today_csv_path()
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        if user_name not in df['Student_Name'].values:
            new_row = {
                'Student_Name': user_name,
                'Registration_Status': 'Active',
                'Last_Seen_Time': 'Never',
                'Attendance_Status': 'Absent'
            }
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            df.to_csv(csv_path, index=False)
            print(f'📊 Added "{user_name}" to today\'s attendance sheet')


# ── RUN REGISTRATION ──────────────────────────────────────────────────────────
folder_to_load = r'C:\Users\User\Downloads\face_demo\deployment\register_manul\2'
images_to_load = (
    glob.glob(os.path.join(folder_to_load, '*.jpeg')) +
    glob.glob(os.path.join(folder_to_load, '*.jpg'))  +
    glob.glob(os.path.join(folder_to_load, '*.png'))
)
register_user_from_files('Soklin', images_to_load)



--- Registering: Soklin ---
  ✅  t1.jpeg  →  emb shape (512,)
  ✅  t2.jpeg  →  emb shape (512,)
  ✅  t3.jpeg  →  emb shape (512,)

✨ Saved 512-dim template for "Soklin" → C:\Users\User\Downloads\face_demo\deployment\database\Soklin.npy


**Registrater to System by scanning**

In [23]:
def register_new_user(user_name):
    """
    Register a new user via webcam.
    Press 's' to capture a sample, 'q' to save and exit.
    Extracts 512-dim embeddings from fine-tuned backbone.
    """
    print(f'Opening camera for: {user_name}')
    print('  Press S to capture sample (take 3-5 photos)')
    print('  Press Q to save and exit')

    cap       = cv2.VideoCapture(0)
    collected = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.putText(frame,
            f'Registering: {user_name}  |  Captured: {len(collected)}',
            (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 100, 0), 2)
        cv2.imshow('Registration', frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('s'):
            aligned = align_and_crop_face_from_frame(frame)
            if aligned is None:
                print('  ⚠️  No face detected — try again')
                continue
            emb = get_embedding(aligned)   # 512-dim
            collected.append(emb)
            print(f'  ✅  Sample #{len(collected)} captured')
        elif key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    if not collected:
        print('Registration closed — no samples saved.')
        return

    master = np.mean(collected, axis=0)
    master = master / np.linalg.norm(master)
    save_path = os.path.join(DATABASE_DIR, f'{user_name}.npy')
    np.save(save_path, master)
    print(f'✨ Saved 512-dim template for "{user_name}" → {save_path}')

    # Add to today's CSV
    csv_path = get_today_csv_path()
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        if user_name not in df['Student_Name'].values:
            new_row = {
                'Student_Name': user_name,
                'Registration_Status': 'Active',
                'Last_Seen_Time': 'Never',
                'Attendance_Status': 'Absent'
            }
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            df.to_csv(csv_path, index=False)
            print(f'📊 Added "{user_name}" to today\'s attendance sheet')


# ── RUN ───────────────────────────────────────────────────────────────────────
register_new_user('Sokha_NewStudent')


Opening camera for: Sokha_NewStudent
  Press S to capture sample (take 3-5 photos)
  Press Q to save and exit
Registration closed — no samples saved.


# **2. Live Attendance & Recognition Monitor**

In [4]:
from datetime import datetime

# ── Load gallery and today's attendance ONCE before the loop ─────────────────
registered_users = load_registered_database()
df = get_or_create_today_attendance(registered_users)
csv_path = get_today_csv_path()

COSINE_THRESHOLD = 0.6   

cap = cv2.VideoCapture(0)
print('\nRecognition started — press Q to stop')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ── Face detection + embedding ────────────────────────────────────────────
    aligned = align_and_crop_face_from_frame(frame)
    if aligned is None:
        # No face in frame — show waiting message and continue
        cv2.putText(frame, 'No face detected', (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100, 100, 100), 2)
        cv2.imshow('Face Recognition — Attendance', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    emb = get_embedding(aligned)   # 512-dim L2-normalised from fine-tuned backbone

    # ── Cosine similarity vs gallery ──────────────────────────────────────────
    final_name    = 'UNKNOWN'
    highest_score = -1.0
    display_color = (0, 0, 255)   # red default

    for name, saved_vector in registered_users.items():
        sim = float(np.dot(emb, saved_vector))
        if sim > highest_score:
            highest_score = sim
            final_name    = name

    if highest_score >= COSINE_THRESHOLD:
        display_color = (0, 255, 0)   # green

        # ── Mark attendance — only write CSV when status actually changes ─────
        if final_name in df['Student_Name'].values:
            current_status = df.loc[
                df['Student_Name'] == final_name, 'Attendance_Status'
            ].values[0]

            if current_status != 'Present':
                current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                df.loc[df['Student_Name'] == final_name, 'Attendance_Status'] = 'Present'
                df.loc[df['Student_Name'] == final_name, 'Last_Seen_Time']    = current_time
                df.to_csv(csv_path, index=False)   # written once per person per day
                print(f'✅ Attendance: {final_name} at {current_time}')
    else:
        final_name = 'UNKNOWN'

    # ── Display ───────────────────────────────────────────────────────────────
    cv2.putText(frame, f'User: {final_name}',
                (20, 50),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, display_color, 2)
    cv2.putText(frame, f'Sim : {highest_score:.3f}  (threshold={COSINE_THRESHOLD})',
                (20, 90),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, display_color, 1)
    cv2.putText(frame, date.today().strftime('%Y-%m-%d'),
                (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    cv2.imshow('Face Recognition — Attendance', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Final summary
present = (df['Attendance_Status'] == 'Present').sum()
absent  = (df['Attendance_Status'] == 'Absent').sum()
print(f'\nSession ended — {present} present, {absent} absent')
print(f'CSV saved: {csv_path}')


Gallery loaded: 4 registered user(s)
📅 New attendance sheet created: attendance_2026-05-31.csv
   4 student(s) initialised → Absent

Recognition started — press Q to stop
✅ Attendance: Sokha_NewStudent at 2026-05-31 13:46:01
✅ Attendance: Nika at 2026-05-31 13:46:02

Session ended — 2 present, 2 absent
CSV saved: C:\Users\User\Downloads\face_demo\attendance\attendance_2026-05-31.csv
